# Data-analytiikan perusteet I: pandas, tutkiva analyysi ja visualisointi

Seuraavassa kahdessa notebookissa tutustutaan käytännön data-analytiikkaan avoimen datan avulla. Käytämme työkaluja, jotka ovat erittäin laajalti standardeja data-analytiikan alalla. Tämä materiaali antaa hyvän työkalupakin perusanalytiikan suorittamiseen, mutta kirjastoista löytyy myös todella paljon syvällisempiäkin menetelmiä. 

## Jupyter Notebook lyhyesti

Jupyter Notebook on työkirja, jossa teksti, Python-koodi ja koodin tuottamat tulokset ovat samassa tiedostossa. Sisältö on jaettu **soluihin**, tekstisolut sisältävät ohjeita ja koodisoluissa suoritetaan ohjelmakoodia.

- Valitse koodisolu napsauttamalla sitä. **Shift + Enter** suorittaa solun ja siirtää valinnan seuraavaan soluun. Tulokset, kuten taulukot ja kuvaajat, näkyvät solun alapuolella.
- Etene koodisoluissa **ylhäältä alas**, sillä myöhemmät solut käyttävät aiemmissa soluissa luotuja muuttujia. Kun muutat koodia, suorita solu ja siitä riippuvat solut uudelleen.
- Voit muokata tekstisolua kaksoisnapsauttamalla sitä. **Shift + Enter** näyttää tekstin jälleen muotoiltuna.
- Koodia suorittava **ydin (kernel)** pitää muuttujat muistissa. Jos käynnistät ytimen uudelleen, suorita myös aiemmat koodisolut uudelleen.
- Tallenna muokkauksesi **Ctrl + S** -näppäinyhdistelmällä. Notebook tallentuu `.ipynb`-tiedostona.

## Oppimistavoitteet

Kun olet käynyt notebookin läpi, osaat:

- lukea taulukkomuotoista dataa pandas-DataFrameen
- tarkistaa aineiston rakenteen ja tietotyypit sekä tarkastella muuttujien jakaumia
- valita ja suodattaa havaintoja
- tunnistaa puuttuvia arvoja ja toistuvia rivejä eli duplikaatteja
- luoda uusia muuttujia laskemalla ja `.apply()`-metodilla
- etsiä tekstistä merkkijonoja säännöllisillä lausekkeilla (regex)
- piirtää histogrammeja, pylväskaavioita, hajontakuvioita ja laatikkokuvioita seaborn-kirjastolla
- tulkita kuvaajia analyysin näkökulmasta

## Avoin data ja lisenssi

Käytämme opetusta varten rajattua otosta HSL:n **Helsingin ja Espoon kaupunkipyörillä ajetut matkat** -aineistosta. Alkuperäinen data sisältää matkojen lähtö- ja pääteasemat, ajat, pituudet ja kestot.

- **Ylläpitäjä:** Helsingin seudun liikenne (HSL)
- **Alkuperäinen tekijä / datan omistaja:** City Bike Finland
- **Lisenssi:** Creative Commons Nimeä 4.0 (CC BY 4.0)
- **Lähde:** https://hri.fi/data/fi/dataset/helsingin-ja-espoon-kaupunkipyorilla-ajatut-matkat

Tässä notebookissa käytetään tiedostoon `hsl_matkat_2021_07_otos.csv` tallennettua 39 rivin otosta heinäkuun 2021 aineistosta.

## 1. Kirjastot ja aineiston lukeminen

**pandas** on kirjasto taulukkomuotoisen datan käsittelyyn. **seaborn**- ja **matplotlib**-kirjastoilla piirretään kuvaajia. Luemme CSV-tiedoston `pd.read_csv()`-funktiolla.

Ennen kuin voimme suorittaa mitään syvällisempiä analyysimenetelmiä, tulee ymmärtää, millaista dataa ylipäätään käsitellään. Onko pyörällä kuljettu matka metreissä kokonaislukuna vai kilometreinä liukulukuna? Useat datankeräysmenetelmät tallentavat tietoa muodossa, jonka tulkitseminen voi olla ihmiselle epäintuitiivista. 
Tällaisissa tapauksissa voisimme hyödyntää luvun 7 menetelmiä. Tässä aineistossa matkat ovat kuitenkin metreinä ja kestot sekunteina, joten voimme aloittaa rivien tarkastelusta.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Luetaan CSV-tiedoston sisältö DataFrame-taulukoksi.
df = pd.read_csv('hsl_matkat_2021_07_otos.csv')
print(f"Aineiston koko: {df.shape[0]} riviä × {df.shape[1]} saraketta")

# Näytetään taulukon ensimmäiset viisi riviä.
df.head()

### Mikä on DataFrame?

DataFrame muistuttaa Excel-taulukkoa: **rivit** ovat havaintoja ja **sarakkeet** muuttujia. Tässä yksi rivi vastaa yhtä kaupunkipyörämatkaa.

Tuloste kertoo, että aineistossa on 39 riviä ja 8 saraketta. `head()` näyttää niistä vain ensimmäiset viisi riviä. Seuraavaksi tarkistamme sarakkeiden nimet ja tietotyypit, jotta tiedämme, mitä arvoilla voidaan tehdä.

In [ ]:
print("Sarakkeet:")
print(df.columns.tolist())
print("\nTietotyypit:")
print(df.dtypes)

<span style="color:red">Huomaa,</span> että lähtö- ja paluuajat ovat tässä tulosteessa `object`-tyyppisiä, tässä tapauksessa tekstiä. Aikoja voidaan tarkastella sellaisinaan, mutta niiden välisten aikaerojen laskeminen edellyttäisi tyyppien muuntamista päivämäärä- ja aikatyypiksi.

## 2. Sarakkeiden nimeäminen selkeämmiksi

Tämän raakadatan sarakkeiden nimet ovat englanniksi ja vähän pidemmän puoleisia. Esimerkiksi pitkien tai sekavien sarakenimien tapauksessa on usein hyödyllistä nimetä ne lyhyemmiksi ja yhdenmukaisiksi. Tämä ei muuta havaintoja, ainoastaan sarakkeiden nimiä.

In [ ]:
df = df.rename(columns={
    'Departure': 'lahto_aika',
    'Return': 'paluu_aika',
    'Departure station id': 'lahtoasema_id',
    'Departure station name': 'lahtoasema',
    'Return station id': 'paluuasema_id',
    'Return station name': 'paluuasema',
    'Covered distance (m)': 'matka_m',
    'Duration (sec.)': 'kesto_s'
})

df.head()

Taulukossa näkyvät nyt lyhyemmät sarakenimet. Esimerkiksi matkan pituus löytyy nimellä `matka_m`, ja sen arvot ovat edelleen metrejä.

## 3. Rakenteen tutkiminen

Tarkistamme aineiston koon ja arvojen vaihtelun, jotta mahdolliset puutteet tai poikkeamat huomataan ennen tarkempaa analyysiä. Kolme hyödyllistä perustyökalua ovat:

- `df.shape` → rivien ja sarakkeiden määrä
- `df.info()` → sarakkeet, tietotyypit ja arvojen määrät ilman puuttuvia arvoja
- `df.describe()` → numeeristen sarakkeiden tunnuslukuja

In [ ]:
print("shape:", df.shape)
print("\ninfo():")
df.info()

In [ ]:
df[['matka_m', 'kesto_s']].describe()

### Miten `describe()`-tulosta luetaan?

- **count** kertoo, montako havaintoa laskennassa oli mukana.
- **mean** on keskiarvo.
- **50%** on mediaani: vähintään puolet havainnoista on enintään mediaanin suuruisia ja vähintään puolet vähintään mediaanin suuruisia.
- **min** ja **max** auttavat löytämään poikkeavia tai epäilyttäviä arvoja.

Jo tässä vaiheessa voidaan huomata poikkeamia kuten esimerkiksi nollan metrin matka. Se voi olla todellinen käyttötilanne, mittausvirhe tai muu poikkeus. Säilytämme sen toistaiseksi aineistossa ja käsittelemme datan siivousta tarkemmin toisessa notebookissa.

`info()`-tuloksen jokaisessa sarakkeessa on 39 arvoa, joten puuttuvia arvoja ei tässä otoksessa näy. `describe()`-taulukon `count` vahvistaa saman matkan pituudelle ja kestolle.

## 4. Puuttuvat arvot ja duplikaatit

Aineistosta tulisi tarkistaa ainakin puuttuvat arvot ja keskenään samanlaiset rivit eli duplikaatit. Puuttuvat arvot voivat jättää havaintoja laskennan ulkopuolelle, ja duplikaatit voivat kasvattaa matkojen määriä virheellisesti. `isna()` merkitsee puuttuvat arvot totuusarvolla `True`. Kun tulokseen sovelletaan `sum()`-metodia, saadaan puuttuvien arvojen määrä kussakin sarakkeessa.

In [ ]:
print("Puuttuvat arvot sarakkeittain:")
print(df.isna().sum())
print("\nTäysin identtisiä duplikaattirivejä:", df.duplicated().sum())

Tulosteen nollat kertovat, ettei tässä pienessä otoksessa ole puuttuvia arvoja eikä keskenään samanlaisia rivejä. Luomme siksi erillisen esimerkkitaulukon, johon lisäämme ensimmäisen rivin toistamiseen. Näin näemme käytännössä, miten duplikaatti tunnistetaan ja poistetaan.

In [ ]:
demo = pd.concat([df, df.iloc[[0]]], ignore_index=True) # Haetaan alkuperäisen dataframen ensimmäinen rivi, ja lisätään se samaan dataframeen. Kopio tallennetaan demo-muuttujaan.
print("Rivejä ennen poistoa:", len(demo))
print("Duplikaatteja:", demo.duplicated().sum())

demo = demo.drop_duplicates()
print("Rivejä poiston jälkeen:", len(demo))

Rivimäärä kasvaa ensin 40:een ja palautuu poiston jälkeen 39:ään. `duplicated()` laskee ylimääräisen kopion, joten duplikaattien määrä on yksi.

## 5. Yksittäisten sarakkeiden tutkiminen

Kategorinen muuttuja kuvaa esimerkiksi luokkaa tai nimeä, kuten lähtöasemaa. `value_counts()` laskee, kuinka monta kertaa kukin arvo esiintyy sarakkeessa. Se vastaa esimerkiksi kysymykseen: **miltä asemilta tämän otoksen matkat lähtivät useimmin?**

In [ ]:
df['lahtoasema'].value_counts().head(10)

Listan ensimmäisenä on Abraham Wetterin tie, jolta lähti tässä otoksessa viisi matkaa.

`nunique()` kertoo erilaisten arvojen lukumäärän. `unique()` taas näyttää itse arvot.

In [ ]:
print("Erilaisia lähtöasemia:", df['lahtoasema'].nunique())
print("Ensimmäiset 10 asemanimeä:")
print(df['lahtoasema'].unique()[:10])

Lukumäärä kertoo, kuinka monta eri lähtöasemaa otoksessa on. `unique()` näyttää nimet esiintymisjärjestyksessä, joten sen ensimmäiset kymmenen nimeä eivät ole yleisyysjärjestyksessä.

## 6. Datan valitseminen ja suodattaminen

Usein koko DataFramea ei tarvitse käsitellä kerralla. Voit valita taulukosta osan esimerkiksi näin:

- `df['sarake']` → yksi sarake
- `.loc[...]` → valinta rivitunnisteiden, sarakkeiden nimien tai ehtojen avulla
- `.iloc[...]` → valinta rivien ja sarakkeiden sijainnin perusteella

Seuraavissa esimerkeissä rajaamme näkyviin vain muutaman rivin ja sarakkeen, jotta taulukkoa on helpompi lukea.

In [ ]:
# Ensimmäiset viisi matkaa: vain lähtöasema ja matkan pituus
df.loc[:4, ['lahtoasema', 'matka_m']] 

# a:b määräävät sisällytettävät rivit. 
# [a:] = Kaikki rivit a:sta lähtien
# [:b] = Kaikki rivit b:hen asti
# [:] = Kaikki rivit

In [ ]:
# Ensimmäiset 3 riviä ja ensimmäiset 4 saraketta
df.iloc[:3, :4]

Ensimmäinen valinta näyttää viisi riviä ja kaksi saraketta, toinen kolme riviä ja neljä saraketta. `.loc[:4]` sisältää rivitunnisteen 4, kun taas `.iloc[:3]` jättää sijainnin 3 pois. Tässä rivitunnisteet ovat järjestyksessä 0, 1, 2 ja niin edelleen.

### Ehtoon perustuva suodatus

Vertailu tuottaa jokaiselle riville totuusarvon `True` (tosi) tai `False` (epätosi). Kun sijoitamme ehdon DataFramen hakasulkeisiin, mukaan jäävät vain rivit, joilla ehto toteutuu.

Rajataan esimerkiksi mukaan vähintään kolmen kilometrin matkat, jotta voimme tarkastella niitä erikseen.

In [ ]:
pitkat = df[df['matka_m'] >= 3000]
# Näemme vertailussa rivien 0-3 olevan False, ja rivin 4 olevan ensimmäinen ehdon täyttävä rivi.
print((df['matka_m'] >= 3000).head(5))
print("Vähintään 3 km matkoja:", len(pitkat))
pitkat[['lahtoasema', 'paluuasema', 'matka_m']].head()

Tulostettu määrä kertoo kaikkien ehdon täyttävien matkojen lukumäärän. Koodisolun taulukko näyttää niistä vain ensimmäiset viisi. Rivien alkuperäiset tunnisteet säilyvät suodatuksessa.

## 7. Uusien muuttujien luominen

Alkuperäiset muuttujat eivät aina ole analyysiin sopivimmassa muodossa. Voimme esimerkiksi muuntaa metrit kilometreiksi ja sekunnit minuuteiksi laskemalla suoraan sarakkeiden arvoilla.

Laskemme myös keskinopeuden jakamalla kilometrit tunneiksi muunnetulla kestolla.

In [ ]:
df['matka_km'] = df['matka_m'] / 1000
df['kesto_min'] = df['kesto_s'] / 60
df['nopeus_kmh'] = df['matka_km'] / (df['kesto_s'] / 3600)

df[['matka_m', 'matka_km', 'kesto_s', 'kesto_min', 'nopeus_kmh']].head()

Ensimmäisen rivin 1602 metriä on nyt 1,602 kilometriä. Matkan pituus ja kesto esitetään uusissa sarakkeissa eri yksiköissä. Keskinopeus kuvaa koko matkaa, joten mahdolliset pysähdykset vaikuttavat siihen.

### `.apply()` ja oma funktio

Jos muunnos perustuu ehtoihin, voimme kirjoittaa tavallisen Python-funktion ja soveltaa sitä jokaiseen sarakkeen arvoon `.apply()`-metodilla.

Luokittelu helpottaa eripituisten matkojen määrien ja kestojen vertailua. Voisimme luokitella matkat sanallisesti esimerkiksi näin:

- alle 1,5 km → `lyhyt`
- vähintään 1,5 km mutta alle 3 km → `keskipitkä`
- vähintään 3 km → `pitkä`

In [ ]:
def luokittele_matka(km):
    if km < 1.5:
        return 'lyhyt'
    elif km < 3:
        return 'keskipitkä'
    else:
        return 'pitkä'

df['pituusluokka'] = df['matka_km'].apply(luokittele_matka)
df[['matka_km', 'pituusluokka']].head(10)

In [ ]:
df['pituusluokka'].value_counts()

Tässä otoksessa lyhyitä matkoja on 11, keskipitkiä 14 ja pitkiä 14. Luokkarajat ovat itse valittuja: niiden muuttaminen muuttaisi myös luokkien matkamääriä.

## 8. Säännölliset lausekkeet eli regex

Säännöllinen lauseke eli **regex** määrittelee, millaisia merkkijonoja tekstistä etsitään. Asemien nimissä metroasema on merkitty tekstillä `(M)`. Sulut ovat regexissä erikoismerkkejä, joten ne suojataan kenoviivalla: `\(M\)`.

pandas osaa etsiä lauseketta vastaavia merkkijonoja suoraan tekstisarakkeesta `.str.contains()`-metodilla.

Jos aihe kiinnostaa enemmän, voi asiaan perehtyä syvällisemmin Pythonin dokumentaatiossa: https://docs.python.org/3/howto/regex.html ja pandasin dokumentaatiossa: https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html

Näissä tehtävissä pärjäämme tällä lyhyellä esittelyllä. 

In [ ]:
metro_lahdot = df['lahtoasema'].str.contains(r'\(M\)', regex=True)
print("Metroasemalta alkaneita matkoja:", metro_lahdot.sum())
df.loc[metro_lahdot, ['lahtoasema', 'paluuasema']]

Haku löysi kaksi metroasemalta alkanutta matkaa. Taulukosta näemme, mitkä matkat täyttivät ehdon.

Voimme yhdistää kaksi tekstihakua `|`-operaattorilla ja etsiä matkat, joiden **lähtö- tai paluuasema** on metroasema. Mukaan tulevat myös matkat, joiden molemmat asemat ovat metroasemia.

In [ ]:
metro_matka = (
    df['lahtoasema'].str.contains(r'\(M\)', regex=True) |
    df['paluuasema'].str.contains(r'\(M\)', regex=True)
)
print("Matkoja, joiden lähtö- tai paluuasema on metroasema:", metro_matka.sum())

Yhdistetty ehto löytää kolme matkaa. Sama rivi lasketaan vain kerran, vaikka sekä lähtö- että paluuasema täyttäisivät ehdon.

# Osa II - Tutkiva data-analyysi kuvaajien avulla

Tutkivassa data-analyysissä kuvaajien avulla tarkastellaan muuttujien jakaumia, etsitään poikkeavia havaintoja ja tutkitaan muuttujien välisiä yhteyksiä.

## 9. Histogrammi - numeerisen muuttujan jakauma

Histogrammi näyttää, mille arvoalueille havainnot sijoittuvat. X-akselilla on muuttujan arvo ja pylvään korkeus kertoo havaintojen määrän.

Katsotaan, painottuvatko matkat lyhyisiin vai pitkiin matkoihin. `bins=10` jakaa matkojen pituudet kymmeneen luokkaväliin.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x='matka_km', bins=10, color='blue')
plt.title('Kaupunkipyörämatkojen pituusjakauma')
plt.xlabel('Matkan pituus (km)')
plt.ylabel('Matkojen määrä')
plt.show()

**Tulkinta:** suurin osa tämän pienen otoksen matkoista on muutaman kilometrin mittaisia. Ensimmäiseen pylvääseen sisältyy myös aiemmin löydetty nollan metrin matka. Yksittäistä nolla-arvoa ei kuitenkaan voi erottaa samassa luokkavälissä olevista muista lyhyistä matkoista.

## 10. Pylväskaavio - havaintojen määrä kategorioittain

Pylväskaaviolla vertailemme lähtöasemien matkamääriä. Kaikkien lähtöasemien näyttäminen tekisi kuvaajasta sekavan. Siksi valitaan ensin viisi yleisintä asemaa piirrettäväksi.

In [ ]:
# Valitaan viisi yleisintä asemaa ja asetetaan ne kuva_data-muuttujaan kuvaajan piirtämistä varten
top5 = df['lahtoasema'].value_counts().head(5).index
kuva_data = df[df['lahtoasema'].isin(top5)]

# Määrittää kuvan koon
plt.figure(figsize=(10, 5)) 

# Parametri y määrittää, minkä sarakkeen mukaan y-akseli piirretään
# order ottaa tässä parametrina listan halutusta järjestyksestä 
sns.countplot(data=kuva_data, y='lahtoasema', order=top5, color='brown')
plt.title('Viisi yleisintä lähtöasemaa opetusaineistossa')
plt.xlabel('Matkojen määrä')
plt.ylabel('Lähtöasema')
plt.show()

**Tulkinta:** Abraham Wetterin tie erottuu viidellä lähdöllä. Pylvään pituus kertoo matkojen määrän, joten pidempi pylväs tarkoittaa useampia lähtöjä. Tulos kuvaa tätä pientä otosta, eikä siitä voi päätellä koko pyöräkauden yleisintä lähtöasemaa.

## 11. Hajontakuvio - kahden numeerisen muuttujan suhde

Hajontakuviossa jokainen piste on yksi havainto. Katsotaan, miten matkan pituus ja kesto liittyvät toisiinsa.

Värit erottavat aiemmin luodut pituusluokat toisistaan.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='matka_km', y='kesto_min', hue='pituusluokka')
plt.title('Matkan pituus ja kesto')
plt.xlabel('Matkan pituus (km)')
plt.ylabel('Kesto (min)')
plt.show()

**Tulkinta:** pidempi matka kestää yleensä kauemmin, mutta kesto vaihtelee myös samanpituisilla matkoilla. Esimerkiksi pysähdykset, ajonopeus ja mittauspoikkeamat voivat vaikuttaa kestoon.

## 12. Laatikkokuvio - jakauma ryhmittäin

Laatikkokuvio eli boxplot tiivistää jakauman keskeiset piirteet. Laatikon sisällä oleva viiva näyttää mediaanin, ja laatikko sisältää aineiston keskimmäiset 50 % havainnoista. Kuvion avulla voidaan tarkastella myös havaintojen hajontaa (ns. laatikon viikset) ja mahdollisia poikkeavia havaintoja. Se soveltuu hyvin esimerkiksi matkojen kestojen vertailuun eri pituusluokkien välillä.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='pituusluokka', y='kesto_min', order=['lyhyt', 'keskipitkä', 'pitkä'])
plt.title('Matkan kesto pituusluokittain')
plt.xlabel('Pituusluokka')
plt.ylabel('Kesto (min)')
plt.show()

Laatikkokuvion avulla voidaan tunnistaa helposti mahdollisia poikkeavia havaintoja. Keskipitkien matkojen kohdalla kaksi havaintoa sijoittuu selvästi muun jakauman ulkopuolelle. Sama ilmiö oli havaittavissa myös hajontakuviossa, mutta laatikkokuvio tuo sen selkeämmin esiin.

Poikkeava piste ei vielä tarkoita virhettä, vaan matkan tietoja kannattaa tutkia tarkemmin.

## 13. Korrelaatio ja lämpökartta

Korrelaatio mittaa kahden numeerisen muuttujan lineaarista yhteyttä. Arvo on välillä -1 ja +1. Lähellä +1 oleva arvo tarkoittaa vahvaa samansuuntaista yhteyttä ja lähellä -1 oleva arvo vahvaa vastakkaissuuntaista yhteyttä.

Korrelaatio **ei yksin osoita syy-seuraussuhdetta**.

Lähellä nollaa oleva arvo kertoo heikosta lineaarisesta yhteydestä. Laskemme korrelaatiot, jotta voimme verrata matkan pituuden, keston ja keskinopeuden yhteyksiä myös lukuarvoina. Lämpökartta esittää saman taulukon väreinä.

In [ ]:
num = df[['matka_m', 'kesto_s', 'nopeus_kmh']]
korrelaatiot = num.corr()
korrelaatiot

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(korrelaatiot, annot=True, fmt='.2f') # "annot=True" Annotoi korrelaation määrän alkioiden päälle. .2f määrittää kuinka monella desimaalilla korrelaatio esitetään.
plt.title('Numeeristen muuttujien korrelaatiot')
plt.show()

**Tulkinta:** matkan pituuden ja keston korrelaatio on noin 0,72, mikä tukee hajontakuvion havaintoa: pidemmät matkat kestävät yleensä kauemmin. Lävistäjän ykköset ovat muuttujien korrelaatioita itsensä kanssa. Keskinopeus on laskettu pituudesta ja kestosta. Tämä kannattaa huomioida, kun tulkitsemme nopeuden korrelaatioita.

# Harjoitukset

Vastaa kysymyksiin tässä notebooksissa esitettyjen menetelmien avulla. Voit käyttää alla olevaa tyhjää solua omiin laskuihisi. Vastaukset ovat joko **kokonaislukuja** tai **merkkijonoja**.

**Tehtävät 1-3** harjoittavat tulosten lukemista: vastaukset löytyvät suoraan aiempien koodiesimerkkien tuloksista. **Tehtävät 4-7** vaativat soveltamista: yhdistä aiemmin läpikäytyjä komentoja ja kirjoita oma ratkaisusi. Vihjeitä löydät tarvittaessa yhteenvedon jälkeen, yritä kuitenkin ratkaista tehtävät ensin ilman.

1. Kuinka monta matkaa aineistossa on?  
2. Kuinka monta erilaista lähtöasemaa aineistossa on?  
3. Mikä lähtöasema esiintyy useimmin? Kirjoita aseman nimi täsmälleen datan mukaisesti.  
4. Miltä asemalta aineiston pisin matka lähti?  
5. Kuinka monelta eri lähtöasemalta vähintään 20 minuuttia kestäneet matkat lähtivät?  
6. Mikä on vähintään 3000 metriä pitkien matkojen keston mediaani sekunteina? Anna vastaus kokonaislukuna.  
7. Kuinka monta metriä ajettiin yhteensä matkoilla, joiden lähtö- tai paluuaseman nimessä esiintyy merkintä `(M)`? Laske kukin matka mukaan vain kerran ja anna vastaus kokonaislukuna.

In [ ]:
# Kirjoita oma koodisi tähän.

## Yhteenveto

Tässä notebookissa tutustuimme aineiston rakenteeseen, suodatimme havaintoja, loimme uusia muuttujia ja tarkastelimme aineistoa kuvaajien avulla. Hyvä analyysi alkaa datan rakenteen ja laadun ymmärtämisestä. Sen pohjalta tunnuslukuja ja kuvaajia on helpompi tulkita.

Seuraavassa notebookissa harjoittelemme ryhmittelyä, pivot-taulukoiden muodostamista, taulukoiden yhdistämistä ja datan järjestelmällistä siivousta.

## Vihjeet harjoituksiin

4. **Pisimmän matkan lähtöasema:** selvitä suurin matkan pituus `describe()`-tuloksen `max`-riviltä. Rajaa sitten aineistosta tämän pituiset matkat ja tarkastele `lahtoasema`-saraketta. Voit käyttää suodatuksessa `>=`-vertailua, sillä mikään matka ei ylitä suurinta pituutta.
5. **Eri lähtöasemien määrä rajatussa aineistossa:** käytä `kesto_min`-saraketta, ehtoon perustuvaa suodatusta ja `nunique()`-metodia.
6. **Keston mediaani rajatussa aineistossa:** suodata matkan pituuden perusteella ja lue `kesto_s`-sarakkeen mediaani `describe()`-tuloksen `50%`-riviltä.
7. **Metroasemaan liittyvien matkojen yhteispituus:** yhdistä `.str.contains()`-haut `|`-operaattorilla, suodata matkat ja laske `matka_m`-sarakkeen arvot yhteen `sum()`-metodilla.
